In [1]:
import pandas as pd
from tkinter import messagebox
import pyperclip
import re
from datetime import timedelta, datetime

In [ ]:
# 1 Функция парсинга результатов
def parse_battle_stats():
    imported_game_log = pyperclip.paste()
    if not imported_game_log.strip():
        print("❌ Буфер обмена пуст. Скопируй статистику боя и запусти скрипт снова.")
        return None

    # --- Результат: Победа / Поражение ---
    result_match = re.search(r'(Победа|Поражение) в миссии', imported_game_log)
    result = result_match.group(1) if result_match else "Неизвестно"

    # --- Название миссии ---
    mission_match = re.search(r'миссии\s+"([^"]+)"', imported_game_log)
    mission = mission_match.group(1) if mission_match else "Неизвестно"

    # --- Итого: СЛ, СОИ (FRP), ОИ (RP) — только последнее вхождение ---
    total_matches = re.findall(r'Итого:\s*(\d+)\s*СЛ,\s*(\d+)\s*СОИ,\s*(\d+)\s*ОИ', imported_game_log)
    if not total_matches:
        print("❌ Не удалось найти ни одного вхождения 'Итого'.")
        return None

    # Берём ПОСЛЕДНЕЕ вхождение (финальные итоги)
    last_match = total_matches[-1]
    total_sl = int(last_match[0])   # Silver Lions
    total_frp = int(last_match[1])  # Free Research Points
    total_rp = int(last_match[2])   # Research Points

    # --- Очки миссии ---
    mission_points = re.findall(r'(\d+)\s*очк(?:о|а|ов)\s*миссии', imported_game_log)
    total_mission_points = sum(int(x) for x in mission_points)

    # --- Сессия ---
    session_match = re.search(r'Сессия:\s*([a-f0-9]+)', imported_game_log)
    session_id = session_match.group(1) if session_match else None
    if not session_id:
        print("❌ Не удалось найти session_id.")
        return None

    # --- Активность (%) ---
    activity_match = re.search(r'Активность:\s*(\d+)%', imported_game_log)
    activity_percent = int(activity_match.group(1)) if activity_match else None

    # --- Использованная техника ---
    vehicles = set()

    # Паттерн 1: "Время активности" — ищем текст до "Цифры + (ПА)"
    pattern_active = r'^\s*(.+?)\s+\d+\s*\+\s*$$ПА$$'
    # Паттерн 2: "Время игры" — ищем текст до "95% ... 4:51"
    pattern_game = r'^\s*(.+?)\s+\d+%.*?\d+:\d+'

    active_time_matches = re.findall(pattern_active, imported_game_log, re.MULTILINE)
    game_time_matches = re.findall(pattern_game, imported_game_log, re.MULTILINE)
    
    all_vehicles = active_time_matches + game_time_matches

    for v in all_vehicles:
        cleaned = re.sub(r'\s+', ' ', v.strip())
        # Исключаем ложные срабатывания (например, "Заработано", "Итого")
        if cleaned and not re.match(r'^[\[\]"]', cleaned) and len(cleaned) > 1:
            vehicles.add(cleaned)

    vehicles = ", ".join(sorted(vehicles)) if vehicles else "Неизвестно"


    # --- Время миссии ---
    mission_time_match = re.search(r'Время игры\s*(\d+:\d+)', imported_game_log)
    mission_time = mission_time_match.group(1) if mission_time_match else "Неизвестно"
    minutes, seconds = map(int, mission_time.split(':'))
    mission_time = timedelta(minutes=minutes, seconds=seconds)

    # --- Бустеры ---
    boosters_sl_match = re.search(r'Активные усилители на СЛ:[^.]*?Общий:\s*\+\s*(\d+)%СЛ', imported_game_log)
    boosters_rp_match = re.search(r'Активные усилители на ОИ:[^.]*?Общий:\s*\+\s*(\d+)%ОИ', imported_game_log)

    boosters_sl_percent = int(boosters_sl_match.group(1)) if boosters_sl_match else None
    boosters_rp_percent = int(boosters_rp_match.group(1)) if boosters_rp_match else None

    return {
        'session_id': session_id,
        'vehicles': vehicles,
        'total_sl': total_sl,
        'total_frp': total_frp,
        'total_rp': total_rp,
        'total_mission_points': total_mission_points,
        'result': result,
        'mission': mission,
        'activity_percent': activity_percent,
        'mission_time': mission_time,
        'battle_type': 'РБ',
        'max_br': "111",
        'br_country': "Африка",
        'boosters_sl_percent': boosters_sl_percent,
        'boosters_rp_percent': boosters_rp_percent
    }

In [9]:
session_start_time = datetime.now()

df_for_session = df

In [5]:
xlsx_path = r"D:\py\wt_stats\data.xlsx"
df = df = pd.read_excel(xlsx_path, engine='openpyxl')

In [6]:
# 2 Функция сохранения в эксель
def save_to_excel(data, xlsx_path):
    
    global df_for_session

    columns = [
        'session_id', 'vehicles', 'total_sl', 'total_frp', 'total_rp',
        'total_mission_points', 'result', 'mission', 'activity_percent', 
        'mission_time', 'battle_type', 'max_br', 'br_country', 
        'boosters_sl_percent', 'boosters_rp_percent'
    ]

    try:
        df = pd.read_excel(xlsx_path, engine='openpyxl')
    except (FileNotFoundError, ValueError):
        df = pd.DataFrame(columns=columns)

    # Удаляем строку с таким session_id, если есть
    df = df[df['session_id'] != data['session_id']]

    # Добавляем новую
    new_row = pd.DataFrame([data], columns=columns)
    df = pd.concat([df, new_row], ignore_index=True)
    df_for_session = pd.DataFrame([data], columns=columns)
    df_for_session = pd.concat([df_for_session, new_row], ignore_index=True) # формируем второй датафрейм для finish_window

    # Сохраняем
    # df.to_excel(xlsx_path, index=False, engine='openpyxl')
    print(f"\n ✅ Обновлено: {data['session_id']}")

    return df_for_session

In [23]:
# 3.12 Окно на выходе из программы
def finish_window(df):

    # 3.10.1 Среднее время боя за сессию
    mission_avg_time = sum(df_for_session['mission_time'].fillna(0.005))/ df_for_session.shape[0]
    td = pd.to_timedelta(mission_avg_time, unit='D')
    hours = td.components.hours
    minutes = td.components.minutes
    seconds = td.components.seconds
    formatted_time = f"{hours}:{minutes:02d}:{seconds:02d}"

    # 3.10.2 Длительность сессии
    session_end_time = datetime.now()
    session_total_time = session_end_time - session_start_time
    session_total_time = session_total_time.total_seconds()
    hours = int(session_total_time // 3600)
    minutes = int((session_total_time % 3600) // 60)
    session_total_time = f'{hours} ч, {minutes} мин'

    # 3.10.3 Суммы по sl, rp, mp
    session_total_sl = sum(df['total_sl'])
    session_total_sl = f"{session_total_sl:_}".replace("_", " ")
    session_total_rp = sum(df['total_frp'])
    session_total_rp = f"{session_total_rp:_}".replace("_", " ")
    session_total_mp = sum(df['total_mission_points'])
    session_total_mp = f"{session_total_mp:_}".replace("_", " ")

    # 3.10.4 Средние по sl, rp, mp
    session_average_sl = f"{int(df['total_sl'].mean()):_}".replace("_", " ")
    session_average_rp = f"{int(df['total_frp'].mean()):_}".replace("_", " ")
    session_average_mp = f"{int(df['total_mission_points'].mean()):_}".replace("_", " ")

    # 3.11.5 Винрейт
    winrate = df['result'].value_counts()
    winrate = round(winrate.get('Победа', 1) / winrate.sum() * 100, 1)

    # аптиры
    br_bracket = df['br_bracket'].value_counts()
    br_bracket = ', '.join(f"{idx} ({val})\n" for idx, val in br_bracket.items())

    messagebox.showinfo(
        title='Игровая сессия завершена',
        message=f"""
        Продлилась {session_total_time}, боев - {df_for_session.shape[0]}, побед - {winrate} %
        {formatted_time}
        Заработано всего:
        🐱 {session_total_sl} SL
        💡 {session_total_rp} RP
        🌐 {session_total_mp} MP
        Заработано в среднем:
        🐱 {session_average_sl} SL
        💡 {session_average_rp} RP
        🌐 {session_average_mp} MP
        Статистика аптиров:
        {br_bracket}
        """)

In [19]:
a = df['br_bracket'].value_counts()
for text, number in a.items():
    print(f"{text} - {number}")

Even - 2
Вилка BR - 1


In [29]:
result = f', \n'.join(f"{idx} ({val})" for idx, val in a.items())
print(result)

Even (2), 
Вилка BR (1)


In [24]:

    # analyzer.finish_window(df_for_session)
finish_window(df)